# DS_18 PDF Bronze Extraction (population by province and city/municipality)

This notebook extracts the table in the PSA **Updated Projected Mid-Year Population, based on the 2015 POPCEN: 2020–2025** PDF (DS_18) into a Bronze Delta table.

**Why it is needed:** the DS_18 Excel file (`psa_ds_18`, 2015–2025) gives **province totals that include their HUCs** and does not list HUCs separately. The analysis units keep 16 HUCs separate from their provinces (DL-010), so tourism intensity needs HUC populations. Only the PDF lists every city and municipality, including the HUCs. Both files are the same PSA release: their 2020–2025 province totals are identical.

The PDF is read from Google Drive through the existing Unity Catalog connection (the same file `01_ingest_psa` stores as binary in `psa_ds_18_pdf`). Each printed row is kept **exactly as printed** (all values as text), checks are run, and results are logged to the shared control tables from `00_ingestion_setup`.

**Bronze outputs** (a rerun overwrites both tables):

| Table | One row = |
| --- | --- |
| `psa_ds_18_pdf_rows` | one printed table row: region, province, HUC, city or municipality, with its six yearly values (2020–2025) |
| `psa_ds_18_pdf_report_text` | one non-table line (page title, header, notes) |

No geography mapping, HUC splitting, type conversion or missing-value handling happens here. That belongs in Silver.

In [0]:
%pip install pdfplumber==0.11.9

In [0]:
from pyspark.sql import functions as F
import uuid, io, hashlib
from datetime import datetime

# ── Google Drive connection (existing Unity Catalog connection) ──
GOOGLE_DRIVE_CONNECTION = "capstone_gdrive"

# ── Unity Catalog targets ────────────────────────────────────────
CATALOG = "tuklas_dev"
BRONZE_SCHEMA = "02_bronze"
INGESTION_SCHEMA = "01_ingestion"
NOTEBOOK_NAME = "04_ingest_psa_ds18_pdf"
SOURCE_SYSTEM = "PSA"
DATASET_ID = "DS_18"

run_id = f"psa_ds18pdf_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

# ── Source (same Drive file as SOURCE_FILES["DS_18_pdf"] in 01_ingest_psa) ──
PSA_FOLDER = "https://drive.google.com/drive/folders/1hBSj2dHu1ylbEjRjLzzxwMQ9CBXItiQs"
SOURCE_FILES = {
    "DS_18_pdf": {"url": "https://drive.google.com/file/d/1yxMeiZWzEWjnG44knXsKNb0xHTyk3Ysg/view",
                  "file": "DS_18_PSA_Projected_Mid_Year_Population_2020_2025.pdf"},
}

# ── Bronze tables ────────────────────────────────────────────────
TBL_ROWS = "psa_ds_18_pdf_rows"
TBL_TEXT = "psa_ds_18_pdf_report_text"

def bronze_table(name): return f"`{CATALOG}`.`{BRONZE_SCHEMA}`.`{name}`"
def ingestion_table(name): return f"`{CATALOG}`.`{INGESTION_SCHEMA}`.`{name}`"

METADATA_COLS = ["_source_dataset_id", "_source_file", "_source_sheet", "_source_path", "_source_system",
                 "_ingested_at", "_run_id", "_source_sha256"]

def add_bronze_metadata(df, source_file, source_path, source_sha256):
    return (df
        .withColumn("_source_dataset_id", F.lit(DATASET_ID))
        .withColumn("_source_file", F.lit(source_file))
        .withColumn("_source_sheet", F.lit(None).cast("string"))
        .withColumn("_source_path", F.lit(source_path))
        .withColumn("_source_system", F.lit(SOURCE_SYSTEM))
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_run_id", F.lit(run_id))
        .withColumn("_source_sha256", F.lit(source_sha256)))

def previous_sha256(tbl):
    """Hash of the PDF loaded last time, or None if not available."""
    try:
        vals = [r[0] for r in spark.table(bronze_table(tbl)).select("_source_sha256").distinct().collect() if r[0]]
        return vals[0] if vals else None
    except Exception:
        return None

ingestion_summary = []

def record_ingestion(source_file, bronze_tbl, row_count, col_count, status, notes="", source_path=None):
    """Append to the in-memory summary and MERGE one row into file_manifest (same pattern as 01_ingest_psa)."""
    ingestion_summary.append({"source_dataset_id": DATASET_ID, "source_file": source_file, "bronze_table": bronze_tbl,
                              "row_count": row_count, "column_count": col_count, "status": status, "notes": notes})
    row = spark.createDataFrame([(run_id, NOTEBOOK_NAME, DATASET_ID, SOURCE_SYSTEM, source_file, None, source_path,
                                  bronze_tbl, row_count, col_count, status, notes)],
        schema="run_id STRING, notebook_name STRING, source_dataset_id STRING, source_system STRING, source_file STRING, "
               "source_sheet STRING, source_path STRING, destination_table STRING, row_count BIGINT, column_count BIGINT, "
               "status STRING, notes STRING")
    row = row.withColumn("discovered_at", F.current_timestamp()).withColumn("ingested_at", F.current_timestamp())
    row.createOrReplaceTempView("_manifest_row")
    spark.sql(f"""
    MERGE INTO {ingestion_table('file_manifest')} AS target
    USING _manifest_row AS source
    ON target.run_id = source.run_id
       AND target.source_dataset_id = source.source_dataset_id
       AND target.source_file = source.source_file
       AND COALESCE(target.source_sheet, '') = COALESCE(source.source_sheet, '')
       AND target.destination_table = source.destination_table
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """)

def log_ingestion_error(source_file, source_path, destination_table, error_type, error_message, notes=""):
    row = spark.createDataFrame([(run_id, NOTEBOOK_NAME, DATASET_ID, SOURCE_SYSTEM, source_file, None, source_path,
                                  destination_table, error_type, str(error_message)[:4000], None, notes)],
        schema="run_id STRING, notebook_name STRING, source_dataset_id STRING, source_system STRING, source_file STRING, "
               "source_sheet STRING, source_path STRING, destination_table STRING, error_type STRING, "
               "error_message STRING, error_timestamp TIMESTAMP, notes STRING")
    row = row.withColumn("error_timestamp", F.current_timestamp())
    row.write.format("delta").mode("append").saveAsTable(ingestion_table("ingestion_errors"))

def write_table(df, tbl):
    """One source file per table, so a rerun simply overwrites it (no duplicates)."""
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(bronze_table(tbl)))

print(f"Run ID: {run_id}")

## PDF Extraction Functions

Tested on the actual DS_18 PDF (27 pages, 1,733 table rows). Three things these functions handle:

* **Split numbers:** the PDF prints large values with a gap after the first digit (`1 08,771,978`). Characters are assigned to the label or to one of the six year columns **by position**, so these are re-joined (`108,771,978`).
* **Long labels:** some labels run into the first value column (e.g. `Don Victoriano Chiongbian (Don Mariano Marcos)`) or wrap onto two lines. Position-based assignment keeps them whole and wrapped labels are joined.
* **Unformatted values:** a few rows print numbers without thousands separators (e.g. `SORSOGON 845717`). They are kept as printed and noted.

Each row keeps `page_number`, `pdf_line_number` (where the row starts: for a wrapped label, its first line) and `label_x0` (the label's left position, for review). Every non-table line keeps its own page and line. Labels are kept exactly as printed, including case (the PDF prints provinces and most HUCs in capitals) and notes such as `(Capital)`.

Checks run on the extraction:

* every row has six well-formed values
* exactly one `PHILIPPINES` row and 17 region rows (pre-NIR regions, as printed)
* the region rows add up **exactly** to `PHILIPPINES` in every year

In [0]:
import re
import pdfplumber

YEARS = ["2020", "2021", "2022", "2023", "2024", "2025"]
COL_RIGHT = [241, 304, 367, 429, 492, 555]   # right edge (points) of the 2020..2025 value columns
NUM_START_X = 185                             # digits right of this belong to a value column
SKIP_RE = re.compile(r"^(Page \d+ of \d+|Note:.*|Details may not.*|Updated Projected Mid-Year Population.*|Region, Province, and City/Municipality|01 July 2020.*)$")

def _line_text(chars):
    out, last_x1 = [], None
    for c in chars:
        if last_x1 is not None and c["x0"] - last_x1 > 1.5:
            out.append(" ")
        out.append(c["text"])
        last_x1 = c["x1"]
    return re.sub(r"\s+", " ", "".join(out)).strip()

def _col(x1):
    for k, right in enumerate(COL_RIGHT):
        if x1 <= right + 5:
            return k
    return len(COL_RIGHT) - 1

def parse_ds18_pdf(pdf_bytes):
    """Extract every table row of the DS_18 PDF exactly as printed.

    Returns (rows, text_lines, info). Values stay text. Characters are assigned to the label or to one of the
    six year columns by position, so numbers the PDF splits ('1 08,771,978') are re-joined and labels that run
    into the first column are kept whole. Labels printed on two lines are joined, and the row keeps the page and
    line where its label starts. Every text line keeps its own page and line."""
    rows, text_lines, info = [], [], {"pages": 0, "title": None, "note": None}
    with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
        info["pages"] = len(pdf.pages)
        pending = []
        for pno, page in enumerate(pdf.pages, start=1):
            chars = [c for c in page.chars if c["text"].strip()]
            groups = []
            for c in sorted(chars, key=lambda c: (round(c["top"]), c["x0"])):
                if groups and abs(c["top"] - groups[-1][0]) <= 2:
                    groups[-1][1].append(c)
                else:
                    groups.append([c["top"], [c]])
            for lno, (_, cs) in enumerate(groups, start=1):
                cs.sort(key=lambda c: c["x0"])
                full = _line_text(cs)
                if SKIP_RE.match(full):
                    if full.startswith("Updated Projected Mid-Year Population for"):
                        info["title"] = full
                    elif full.startswith("Note"):
                        info["note"] = full
                    text_lines.append({"page_number": pno, "pdf_line_number": lno, "line_text": full})
                    continue
                num_cs = [c for c in cs if c["x0"] >= NUM_START_X and (c["text"].isdigit() or c["text"] == ",")]
                label_cs = [c for c in cs if c not in num_cs]
                if not num_cs:
                    pending.append({"page_number": pno, "pdf_line_number": lno, "line_text": full})
                    continue
                cols = [""] * 6
                for c in num_cs:
                    cols[_col(c["x1"])] += c["text"]
                label = _line_text(label_cs)
                row_page, row_line = pno, lno
                if pending and (not label or label[0] in ")-" or label[0].islower()):
                    # wrapped label: the row starts where its label starts
                    label = (" ".join(t["line_text"] for t in pending) + " " + label).strip()
                    row_page, row_line = pending[0]["page_number"], pending[0]["pdf_line_number"]
                else:
                    text_lines.extend(pending)          # stray text lines keep their own page and line
                pending = []
                rows.append({"page_number": row_page, "pdf_line_number": row_line,
                             "label_x0": round(label_cs[0]["x0"]) if label_cs else None,
                             "area_label": label,
                             **{f"pop_{y}": (v or None) for y, v in zip(YEARS, cols)}})
        text_lines.extend(pending)                      # text after the last table row
    return rows, text_lines, info

REGION_RE = re.compile(r"^(NATIONAL CAPITAL REGION|CAR|REGION\b.*|Region\b.*|CARAGA|MIMAROPA|ARMM|BARMM)$", re.I)
VALUE_RE = re.compile(r"\d{1,3}(,\d{3})*|\d+")

def _to_int(v):
    return int(v.replace(",", ""))

def check_ds18(rows):
    """Reconciliation checks. Returns (critical_errors, notes)."""
    errors, notes = [], []
    if not rows:
        return ["No table rows extracted"], notes
    bad = [r for r in rows if any(r[f"pop_{y}"] is None or not VALUE_RE.fullmatch(r[f"pop_{y}"]) for y in YEARS)]
    if bad:
        errors.append(f"{len(bad)} rows without 6 well-formed values, e.g. page {bad[0]['page_number']} '{bad[0]['area_label']}'")
    nat = [r for r in rows if r["area_label"].upper() == "PHILIPPINES"]
    if len(nat) != 1:
        errors.append(f"Expected 1 PHILIPPINES row, found {len(nat)}")
    # 'Caraga' is both a region (left margin) and a municipality in Davao Oriental (indented)
    regions = [r for r in rows if REGION_RE.match(r["area_label"])
               and not (r["area_label"].upper() == "CARAGA" and (r["label_x0"] or 99) > 26)]
    notes.append(f"{len(regions)} region rows")
    if len(regions) != 17:
        errors.append(f"Expected 17 region rows (pre-NIR), found {len(regions)}")
    if nat and not bad and len(regions) == 17:
        for y in YEARS:
            s, n = sum(_to_int(r[f"pop_{y}"]) for r in regions), _to_int(nat[0][f"pop_{y}"])
            if s != n:
                errors.append(f"{y}: region rows sum to {s:,}, PHILIPPINES is {n:,}")
        if not any(e[:4] in YEARS for e in errors):
            notes.append("region rows sum to PHILIPPINES in all 6 years")
    plain = sum(1 for r in rows for y in YEARS if r[f"pop_{y}"] and "," not in r[f"pop_{y}"] and len(r[f"pop_{y}"]) > 3)
    if plain:
        notes.append(f"{plain} values printed without thousands separators (e.g. SORSOGON)")
    return errors, notes

In [0]:
spark.sql(f"""
INSERT INTO {ingestion_table('ingestion_run_log')}
(run_id, notebook_name, source_system, source_folder, started_at, completed_at,
 files_discovered, files_ingested, files_skipped, files_failed, tables_created, status, notes)
VALUES (:run_id, :nb, :sys, :folder, current_timestamp(), NULL, 0, 0, 0, 0, 0, 'STARTED', 'Notebook execution started')
""", args={"run_id": run_id, "nb": NOTEBOOK_NAME, "sys": SOURCE_SYSTEM, "folder": PSA_FOLDER})
print(f"✓ STARTED record inserted for {run_id}")

## Extract and Load

Read the PDF bytes from Drive, extract rows, run checks, and write to Bronze. If a **critical check fails**, nothing is written: the file is recorded as `NEEDS_REVIEW` in `file_manifest` and the reason goes to `ingestion_errors`.

Each Bronze table is recorded in `file_manifest` **right after it is written**, so the manifest always matches what is in Bronze. If writing the second table fails, the first stays recorded as `INGESTED` and the failure is logged against the table that failed.

In [0]:
src = SOURCE_FILES["DS_18_pdf"]
url, fname = src["url"], src["file"]
YEAR_COLS = [f"pop_{y}" for y in YEARS]
hash_note, source_sha256 = None, None     # also reported in the run log
current_tbl = TBL_ROWS                     # table being written, for the error records
try:
    binary_df = (spark.read.format("binaryFile")
                 .option("databricks.connection", GOOGLE_DRIVE_CONNECTION)
                 .load(url))
    pdf_bytes = binary_df.collect()[0]["content"]
    source_sha256 = hashlib.sha256(pdf_bytes).hexdigest()
    old_sha256 = previous_sha256(TBL_ROWS)
    if old_sha256 is None:
        hash_note = "first load with source hash"
    elif old_sha256 == source_sha256:
        hash_note = "source PDF unchanged since previous load"
    else:
        hash_note = f"SOURCE PDF CHANGED since previous load (previous sha256 {old_sha256[:12]}...); review"

    rows, text_lines, info = parse_ds18_pdf(pdf_bytes)
    errors, notes = check_ds18(rows)

    if errors:
        msg = "; ".join(errors)
        print(f"✗ NEEDS_REVIEW: {msg}")
        log_ingestion_error(fname, url, TBL_ROWS, "EXTRACTION_CHECK_FAILED", msg)
        record_ingestion(fname, TBL_ROWS, 0, 0, "NEEDS_REVIEW", msg, url)
    else:
        check_note = (f"pages={info['pages']}; rows={len(rows)}; {' | '.join(notes)}; "
                      f"sha256={source_sha256[:12]}...; {hash_note}")

        current_tbl = TBL_ROWS
        common = {"report_title": info["title"]}
        row_cols = ["report_title", "page_number", "pdf_line_number", "label_x0", "area_label"] + YEAR_COLS
        row_data = [tuple(None if (v := {**common, **r}.get(c)) is None else str(v) for c in row_cols) for r in rows]
        rows_df = add_bronze_metadata(
            spark.createDataFrame(row_data, schema=", ".join(f"{c} STRING" for c in row_cols)), fname, url, source_sha256)
        write_table(rows_df, TBL_ROWS)
        record_ingestion(fname, TBL_ROWS, spark.table(bronze_table(TBL_ROWS)).count(), len(rows_df.columns),
                         "INGESTED", check_note, url)

        current_tbl = TBL_TEXT
        text_cols = ["page_number", "pdf_line_number", "line_text"]
        text_data = [tuple(None if t.get(c) is None else str(t.get(c)) for c in text_cols) for t in text_lines]
        text_df = add_bronze_metadata(
            spark.createDataFrame(text_data, schema=", ".join(f"{c} STRING" for c in text_cols)), fname, url, source_sha256)
        write_table(text_df, TBL_TEXT)
        record_ingestion(fname, TBL_TEXT, spark.table(bronze_table(TBL_TEXT)).count(), len(text_df.columns),
                         "INGESTED", check_note, url)
        print(f"✓ INGESTED: {len(rows)} table rows, {len(text_lines)} text lines | {check_note}")
except Exception as e:
    print(f"✗ FAILED: {fname} -> {current_tbl} — {e}")
    log_ingestion_error(fname, url, current_tbl, "INGESTION_ERROR", str(e))
    record_ingestion(fname, current_tbl, 0, 0, "FAILED", str(e), url)

## Validate Bronze Tables

Read back both tables and confirm metadata columns, a single source file and hash, row counts that match the manifest, and that every row was written by this run.

Failed validations are **collected, not raised here**: each one is written to `ingestion_errors`, the run log is closed as `FAILED` in the next step, and only then does the notebook raise. So a failed check never leaves the run as `STARTED`.

In [0]:
checks, validation_errors = [], []      # validation_errors: (table, message)

def expect(ok, tbl, msg):
    if not ok:
        validation_errors.append((tbl, msg))
    return ok

try:
    for r in [r for r in ingestion_summary if r["status"] == "INGESTED"]:
        tbl = r["bronze_table"]
        t = spark.table(bronze_table(tbl))
        missing = [c for c in METADATA_COLS if c not in t.columns]
        expect(not missing, tbl, f"Missing metadata columns {missing}")
        if not missing:
            expect(t.filter(F.col("_source_system") != SOURCE_SYSTEM).count() == 0, tbl, "Unexpected _source_system")
            expect(t.filter(F.col("_source_dataset_id") != DATASET_ID).count() == 0, tbl, "Unexpected _source_dataset_id")
            expect(t.filter(F.col("_source_path").isNull() | F.col("_ingested_at").isNull()).count() == 0, tbl,
                   "NULL _source_path or _ingested_at")
            expect(t.select("_source_file").distinct().count() == 1, tbl, "More than one source file")
            expect(t.select("_source_sha256").distinct().count() == 1, tbl, "More than one source hash")
            expect(t.filter(F.col("_run_id") != run_id).count() == 0, tbl, "Rows not written by this run")
        n = t.count()
        ok = expect(n == r["row_count"], tbl, f"Row count {n} differs from manifest ({r['row_count']})")
        checks.append((tbl, n, "ok" if ok else f"count differs from manifest ({r['row_count']})"))
    if any(r["bronze_table"] == TBL_ROWS and r["status"] == "INGESTED" for r in ingestion_summary):
        t = spark.table(bronze_table(TBL_ROWS))
        expect(t.filter(F.col("area_label").isNull() | (F.trim("area_label") == "")).count() == 0, TBL_ROWS, "Blank area_label")
        display(t.filter(F.col("area_label").isin("PHILIPPINES", "QUEZON", "LUCENA CITY (Capital)", "CEBU",
                                                   "Cebu City (Capital)", "BAGUIO CITY", "DAVAO CITY"))
                 .select("page_number", "area_label", *YEAR_COLS))
except Exception as e:
    validation_errors.append((None, f"Validation could not run: {e}"))

for tbl, msg in validation_errors:
    print(f"✗ VALIDATION: {tbl} — {msg}")
    log_ingestion_error(fname, url, tbl, "VALIDATION_FAILED", msg)
for r in ingestion_summary:
    if r["status"] != "INGESTED":
        checks.append((r["bronze_table"], None, r["status"]))
if checks:
    display(spark.createDataFrame(checks, "table STRING, row_count BIGINT, result STRING"))

## Ingestion Summary and Run Log Update

Updates this run's `ingestion_run_log` row from `STARTED` to `SUCCESS`, `PARTIAL_SUCCESS`, or `FAILED`. The notes carry the same source-hash note as `file_manifest` (first load, unchanged, or changed). If the run is `FAILED` (including a failed validation), the run log is updated **first** and the notebook then raises, so the job stops and nothing downstream runs.

In [0]:
manifest_df = spark.table(ingestion_table("file_manifest")).filter(F.col("run_id") == run_id)
statuses = [r["status"] for r in ingestion_summary]
tables_created = len({r["bronze_table"] for r in ingestion_summary if r["status"] == "INGESTED"})
error_count = spark.table(ingestion_table("ingestion_errors")).filter(F.col("run_id") == run_id).count()
files_ingested = 1 if "INGESTED" in statuses else 0
files_failed = 1 if ("FAILED" in statuses or validation_errors) else 0
files_skipped = 1 if "NEEDS_REVIEW" in statuses else 0

if validation_errors or not files_ingested:
    final_status = "FAILED"
elif error_count == 0:
    final_status = "SUCCESS"
else:
    final_status = "PARTIAL_SUCCESS"
final_notes = (f"DS_18 PDF extraction: {files_ingested} file ingested, {files_skipped} needs review, "
               f"{files_failed} failed, {tables_created} Bronze tables, {error_count} errors"
               + (f", {len(validation_errors)} validation failures" if validation_errors else "")
               + (f"; sha256={source_sha256[:12]}...; {hash_note}" if hash_note else "") + ".")

spark.sql(f"""
UPDATE {ingestion_table('ingestion_run_log')}
SET status = :status, completed_at = current_timestamp(),
    files_discovered = 1, files_ingested = :fi, files_skipped = :fs, files_failed = :ff,
    tables_created = :tc, notes = :notes
WHERE run_id = :run_id
""", args={"status": final_status, "fi": files_ingested, "fs": files_skipped, "ff": files_failed,
           "tc": tables_created, "notes": final_notes, "run_id": run_id})

print(f"Run {run_id}: {final_status}\n{final_notes}")
display(spark.table(ingestion_table("ingestion_run_log")).filter(F.col("run_id") == run_id))
display(manifest_df.orderBy("destination_table"))
display(spark.table(ingestion_table("ingestion_errors")).filter(F.col("run_id") == run_id))

if final_status == "FAILED":
    raise RuntimeError(f"Run {run_id} FAILED: {final_notes} See {ingestion_table('ingestion_errors')} for this run_id.")